In [0]:
import pandas as pd
import numpy as np

In [0]:
caso_practico_ventas = spark.read.table("workspace.electiva.caso_practico_ventas").toPandas()

In [0]:

caso_practico_ventas

Se verificó que la tabla tiene un total de 123 filas x 9 columnas 

In [0]:
display(caso_practico_ventas.head(10))
display(caso_practico_ventas.tail(5))

Diagnóstico: Durante la revisión y validación inicial del conjunto de datos, se identificaron discrepancias entre los tipos de datos asignados automáticamente y aquellos definidos como correctos en el diccionario de datos. Adicionalmente, se detectaron registros duplicados en la variable id_transaccion, evidenciándose la repetición del identificador TXN-10011

In [0]:
# Conteo absoluto de nulos 
nulos_abs = caso_practico_ventas.isnull().sum()

In [0]:
# Porcentaje de nulos 
nulos_pct = (caso_practico_ventas.isnull().sum() / len(caso_practico_ventas)) * 100

In [0]:

# Consolidado en un DataFrame 
caso_practico_ventas_nulos = pd.DataFrame({'Total Nulos': nulos_abs, 'Porcentaje (%)': nulos_pct.round(2)}) 
print(caso_practico_ventas_nulos)
     

Al realizar la evaluación de completitud del conjunto de datos, se identificó la presencia de valores ausentes en 3 de las 9 variables analizadas. En contraste, los campos (id_transaccion, fecha, id_cliente, categoria_producto, metodo_pago y region) no presentan registros faltantes y cuentan con una cobertura del 100%

In [0]:
#Calculo de media, mediana, desviación estándar, mínimo y máximo para cantidad y precio_unitario.
columnas_num = ['cantidad', 'precio_unitario']
estadisticas = caso_practico_ventas[columnas_num].agg(['mean', 'median', 'std', 'min', 'max']).T
estadisticas.columns = ['Media', 'Mediana', 'Desviación Estándar', 'Mínimo', 'Máximo']
display(estadisticas)

Al analizar las métricas estadísticas calculadas, se pueden destacar los siguientes resultados:

* Unidades vendidas (cantidad): La variable presenta un comportamiento bastante estable, debido a que la media y la mediana coinciden en 2 unidades. La mayoría de las ventas corresponden a cantidades pequeñas, principalmente entre 1 y 4 unidades, con un valor mínimo de 1 y un máximo de 10 unidades.

*Precios (precio_unitario): Se evidencia una distribución fuertemente influenciada por un valor extremo. La mediana es de 49.99, con una desviación estándar de 924.34. La variación la genera el precio máximo registrado de 9,999.00. Este comportamiento sugiere la existencia de una transacción atípica que genera una dispersión considerable en los datos y que deberá ser revisada durante el proceso de limpieza.

_**DATA CLEANING**_

In [0]:
# 1. Identificar cuántas filas duplicadas existen 
cant_duplicados = caso_practico_ventas.duplicated(subset=['id_transaccion']).sum() 
print(f"Total de registros duplicados: {cant_duplicados}")

In [0]:
# 2. Eliminar registros duplicados conservando el primero 
caso_practico_ventas_clean = caso_practico_ventas.drop_duplicates(subset=['id_transaccion'])

In [0]:
# 3. Verificar la nueva cantidad de filas 
print(f"Filas iniciales: {len(caso_practico_ventas)} | Filas tras eliminar duplicados: {len(caso_practico_ventas_clean)}")


Se detectaron 3 registros duplicados dentro del conjunto de datos inicial, compuesto por 123 filas. Después de realizar el proceso de depuración y eliminar las entradas repetidas, el dataset quedó reducido a 120 registros.

In [0]:
# 1. Ver la frecuencia de valores antes de corregir 
print("Antes de estandarizar:")
print(caso_practico_ventas_clean['metodo_pago'].value_counts()) 


In [0]:
# 2. Reemplazar la variante inconsistente 
caso_practico_ventas_clean['metodo_pago'] = caso_practico_ventas_clean['metodo_pago'].replace({'Tarjeta Credito': 'Tarjeta de Crédito'})

In [0]:
# 3. Ver la frecuencia corregida 
print("\nDespués de estandarizar:") 
print(caso_practico_ventas_clean['metodo_pago'].value_counts())

Después de estandarizar las etiquetas "Tarjeta Credito" y "Tarjeta de Crédito", que inicialmente contaban con 27 y 26 registros, respectivamente, se consolidaron ambas categorías, obteniendo un total de 53 transacciones.

In [0]:
# 1. Calcular la mediana de 'cantidad' y la moda de 'satisfaccion\_cliente' 
mediana_cantidad = caso_practico_ventas_clean['cantidad'].median()
moda_satisfaccion = caso_practico_ventas_clean['satisfaccion_cliente'].mode()[0] 
print(f"Mediana calculada para cantidad: {mediana_cantidad}") 
print(f"Moda calculada para satisfacción: {moda_satisfaccion}") 

In [0]:
# 2. Imputar (rellenar) los valores nulos 
caso_practico_ventas_clean['cantidad'] = caso_practico_ventas_clean['cantidad'].fillna(mediana_cantidad)
caso_practico_ventas_clean['satisfaccion_cliente'] = caso_practico_ventas_clean['satisfaccion_cliente'].fillna(moda_satisfaccion)

In [0]:
# 3. Verificar que ya no existan nulos en estas dos columnas 
print("Nulos restantes en cantidad:", caso_practico_ventas_clean['cantidad'].isnull().sum()) 
print("Nulos restantes en satisfacción:", caso_practico_ventas_clean['satisfaccion_cliente'].isnull().sum())

1. Unidades vendidas (cantidad): Debido a que corresponde a una variable cuantitativa discreta, los 5 valores nulos encontrados fueron reemplazados por la mediana del conjunto de datos, equivalente a 2.0 unidades.
2. Satisfacción del cliente (satisfaccion_cliente): Al ser una variable de tipo ordinal con una escala de 1 a 5, los 2 valores nulos fueron completados utilizando la moda, cuyo valor es 4.0, al corresponder a la calificación registrada con mayor frecuencia entre los compradores.

Una vez realizada la imputación, ambas variables quedaron completamente diligenciadas, con un 100% de datos disponibles y 0 valores nulos.

In [0]:
# 1. Convertir el valor atípico de 9999.0 a NaN para no afectar el promedio 
caso_practico_ventas_clean['precio_unitario'] = caso_practico_ventas_clean['precio_unitario'].replace(9999.0, np.nan)

In [0]:
# 2. Calcular la media de precio por categoría (sin incluir el outlier) e imputar 
caso_practico_ventas_clean['precio_unitario'] = caso_practico_ventas_clean.groupby('categoria_producto')['precio_unitario'].transform( lambda x: x.fillna(x.mean()) )

In [0]:
# 3. Verificar que no queden nulos ni valores atípicos 
print("Nulos restantes en precio_unitario:", caso_practico_ventas_clean['precio_unitario'].isnull().sum()) 
print("Precio máximo corregido:", caso_practico_ventas_clean['precio_unitario'].max())

Después de aplicar la corrección correspondiente, el dataset quedó completamente depurado, con un total de 120 registros únicos, 0 valores nulos en todas las variables y precios consistentes dentro del conjunto de datos.


_**Transformación e Ingeniería de Atributos**_

In [0]:
# 1. Columna derivada del importe total 
caso_practico_ventas_clean['monto_total'] =  caso_practico_ventas_clean['cantidad'] *  caso_practico_ventas_clean['precio_unitario']

In [0]:
# 2. Conversión temporal y extracción de día y mes 
caso_practico_ventas_clean['fecha'] = pd.to_datetime( caso_practico_ventas_clean['fecha'])
caso_practico_ventas_clean['dia_semana'] =  caso_practico_ventas_clean['fecha'].dt.day_name()
caso_practico_ventas_clean['mes'] =  caso_practico_ventas_clean['fecha'].dt.month_name()

In [0]:
# 3. Discretización en rangos de consumo 
def categorizar_monto(monto):
    if monto < 100:
        return 'Bajo'
    elif monto <= 500:
        return 'Medio'
    else:
        return 'Alto'

caso_practico_ventas_clean['rango_monto'] = caso_practico_ventas_clean['monto_total'].apply(categorizar_monto)

# Visualizar primeras filas con los nuevos atributos 
display(caso_practico_ventas_clean[['id_transaccion', 'monto_total', 'dia_semana', 'mes', 'rango_monto']].head())

* monto_total: Se obtuvo mediante la operación cantidad * precio_unitario. La variable presenta una mediana de 99.98 y un promedio de 218.62, con un valor máximo de $5,000.00.

* Variables temporales (dia_semana y mes): Estas variables fueron generadas automáticamente a partir de la columna fecha, después de convertirla al formato correspondiente de fecha.

* Categorización del consumo (rango_monto): Las transacciones fueron agrupadas en tres categorías de acuerdo con el valor de monto_total:
- Bajo (< $100): 69 transacciones (57.5%).
- Medio ($100–500): 43 transacciones (35.8%).
- Alto (> $500): 8 transacciones (6.7%).

_**Agregación y Resumen de Datos**_

In [0]:
# 1. Agrupamiento por Categoría 
resumen_cat = caso_practico_ventas_clean.groupby('categoria_producto').agg( total_ingresos=('monto_total', 'sum'), promedio_unidades=('cantidad', 'mean') ).round(2)

In [0]:
# 2. Tabla Dinámica (Región vs Categoría) 
tabla_dinamica = pd.pivot_table( caso_practico_ventas_clean, values='monto_total', index='region', columns='categoria_producto', aggfunc='sum', fill_value=0 ).round(2)

In [0]:
# 3. Satisfacción por Método de Pago 
satisfaccion_pago = caso_practico_ventas_clean.groupby('metodo_pago')['satisfaccion_cliente'].mean().round(2)
display(resumen_cat)
display(tabla_dinamica)
display(satisfaccion_pago)

* Rendimiento por Categoría: La categoría Hogar concentra los mayores ingresos, con 9,101.29, seguida de Ropa, que alcanza 7,788.69 y presenta además el promedio de compra más elevado, con 2.33 unidades por venta. Por su parte, Deportes registra la menor facturación, con $1,895.42.

* Distribución Regional: De acuerdo con la tabla dinámica, se observan diferencias en la facturación según la región. Este presenta la mayor facturación en la categoría Hogar, con 7,033.36, mientras que Oeste concentra las ventas de Ropa, con un total de 4,096.00.

* Satisfacción del Cliente: En relación con el método de pago, las transacciones realizadas en Efectivo presentan el promedio de satisfacción más alto, con una calificación de 4.08. En contraste, Tarjeta de Crédito registra el promedio más bajo, con 3.51.


**_Selección, Filtrado y Muestreo_**

In [0]:
# 1. Filtrado Multicriterio: Región Norte y Satisfacción >= 4 
filtro_norte_sat = caso_practico_ventas_clean[ (caso_practico_ventas_clean['region'] == 'Norte') & (caso_practico_ventas_clean['satisfaccion_cliente'] >= 4) ]

In [0]:
# 2. Muestreo Estratificado (10% por Región) 
muestra_estratificada = caso_practico_ventas_clean.groupby('region', group_keys=False).apply( lambda x: x.sample(frac=0.10, random_state=42) )

In [0]:
# Mostrar resultados 
print(f"Registros filtrados (Norte & Sat >= 4): {len(filtro_norte_sat)}")
display(filtro_norte_sat.head()) 
print(f"\Tamaño de la muestra estratificada: {len(muestra_estratificada)}")
display(muestra_estratificada)

* Filtrado Multicriterio: Se seleccionaron 12 transacciones correspondientes a la región Norte que presentan un nivel de satisfacción igual o superior a 4.0. Este grupo de registros permite examinar las características de las ventas con mejores niveles de satisfacción dentro de esta región.

* Muestreo Estratificado: Se tomó una muestra equivalente al 10% del dataset, conformada por 11 registros, conservando la distribución proporcional de las ventas de acuerdo con la variable region: 3 registros del Norte y 2 de cada una de las demás regiones. De esta manera, se mantiene la representación geográfica del conjunto original para realizar análisis y pruebas de forma más eficiente.